In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR
import json
from src.fewshot.extractor import LabelTransformConfig, _prepare_label_tokens, _parse_parent_annotations
from src.tokenizer_utils import tokenize, decode
from src import clean_tokens

c:\Users\zakga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Input Document

In [2]:
filename = "1989CanLII1415ONCA"
split = "test"
filepath = Path(DATA_DIR) / "original" / split / f"{filename}.html"

with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()

#### Chunking with the chunker

In [3]:
chunker = "paragraph"  # "paragraph" | "sentence"

from src.chunkers.cache import cache_exists, load_cache
from src.chunkers import ChunkerFactory

if not cache_exists(chunker, split, filename):

    # Load spaCy only if needed
    nlp = None
    if chunker == "sentence":
        import spacy
        nlp = spacy.load("en_core_web_trf")
        print("✅ Model loaded.\n")


    token_chunks = ChunkerFactory.get_chunks(
        html_content, method=chunker, split=split, filename=filename, nlp=nlp
    )

else:
    token_chunks = load_cache(chunker, split, filename)




#### Select the few shot example

In [4]:
fewshot_method = "greedy"   # "greedy" | "random"

with open(FEWSHOT_CACHE_DIR / f"examples_{fewshot_method}.json", "r", encoding="utf-8") as f:
    fewshot_file_content = json.load(f)

fewshot_examples = [(example["example"]["input"], example["example"]["output"]) for example in fewshot_file_content["examples"]]



##### Few Shot processing step

In [5]:
spans_in_context = True
nb_fewshot_examples = 6
allowed_labels = ["decision", "legislation", "secondary sources"]

label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_labels=allowed_labels,
    keep_attributes=["labelname"]
)



# Transform the output in it simplified form
final_fewshot = []
total_output_text = ""
for example in fewshot_examples:
    _, output = example

    output_tokens = tokenize(output)
    transformed_output_tokens = _prepare_label_tokens(output_tokens, label_config)

    if spans_in_context:
        final_fewshot.append((example[0], decode(transformed_output_tokens)))

    if not spans_in_context:

        total_output_text += "|||" + decode(transformed_output_tokens)

if not spans_in_context:
    parents_dict = _parse_parent_annotations(total_output_text)
    for parent_name, annotations in parents_dict.items():

        for annotation in annotations:
            final_fewshot.append((decode(clean_tokens(tokenize(annotation))), annotation))


final_fewshot = final_fewshot[:nb_fewshot_examples]


#### Prompt & Message

Helper fonction to transform a system prompt, and fewshots in a message template role. Three roles are proposed by default : system, user, assistant.
If system is turned to false, the system prompt will be given for the first user's turn

In [6]:
prompt_filename = "decomposed0_long.txt"
with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

In [7]:
from src.models import get_message
user_input =  decode(token_chunks[0])
message = get_message(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=True)

#### Call of the model generate method

In [8]:
from src.models import AssistantFactory
gpt5_2_config= {
        "type": "openai",
        "model_name": "gpt-5.2",
        "temperature": 1,
    }

assistant = AssistantFactory.create_from_config(gpt5_2_config)

In [9]:
generated = assistant.generate(message=message)

#### Chunk verification / reject

In [16]:
from src.output_control.processor import OutputProcessor
from src.output_control.fallback import FallbackHandler 

controller = OutputProcessor()
fallback_handler = FallbackHandler(processor=controller)
with_fallback = True
corrected_generated_tokens, status = controller.process(raw_llm_output=generated,
        original_chunk=token_chunks[0],
        allowed_labels=allowed_labels)

if not status.passed:
        print("Output did not pass verification. Invoking fallback mechanism...")
        corrected_generated_tokens, status =  fallback_handler.handle_failure(
                assistant=assistant,
                corrected_output=corrected_generated_tokens,
                original_chunk=token_chunks[0],
                initial_status=status,
                allowed_labels=allowed_labels,
                fallback_prompt_filename= "fallback.txt"
                )
